# Text Generation with an RNN

So far you have created two kinds of sequential networks. Both densely connected and convolutional neural networks work by feeding data through a series of layers of neurons.

Now you are going to learn about an entirely new kind of network called a Relational Neural Network. Instead of simply using the data from a previous layer, these networks take into account the context of the data. This makes them especially good at data that is sequential, like text or video.

Think about it like this: if you were given a single frame of a ball and told to guess what direction the ball was going, it would be impossible. However, if you were given a video of a ball being thrown through the air, you could use the frames before to figure out where the ball is headed. This is the kind of learning that an RNN does.

##Prepping Text Data


Like all neural networks, the first phase of this process is all about data preparation. In this instance, you will need to create your own inputs and outputs using a text file.

The very first step in this process is to import all the data you need to train the network. For this project, you are going to use recipe text pulled and reformatted from a dataset of recipes. But truly any text data will work, so if you want to try it out with another text file, feel free.

1. Upload [this](https://drive.google.com/file/d/1SuGePtaiSfEA0Exs1ZnjSY89Ie-kP5Qq/view?usp=sharing) textfile to your runtime.
2. Click **Runtime>Change Runtime Type**
3. In the dropdown menu under **Hardware Type** select GPU and then click **Save**
4. Run the cell to import the required libraries.

In [ ]:
import tensorflow as tf
import numpy as np
import os
import time

5. Confirm that you have the name of the recipes.txt file correct in the path_to_file variable
6. Run the cell to open and decode the file, and to see the first 250 characters.

In [ ]:
path_to_file = 'recipes.txt'

text = open(path_to_file, 'rb').read().decode(encoding='utf-8')
print(text[:250])

No-Bake Nut Cookies
1 c. firmly packed brown sugar
 1/2 c. evaporated milk
 1/2 tsp. vanilla
 1/2 c. broken nuts (pecans)
 2 Tbsp. butter or margarine
 3 1/2 c. bite size shredded rice biscuitsIn a heavy 2-quart saucepan, mix brown sugar, nuts, evapo


##Length and Unique Characters

Now that you have the data imported, you will need to figure out how long it is and to create a vocabulary from the data. You might have worked with vocabularies in machine learning before, but essentially this will become the list of all possible inputs and outputs for the network to choose.

1. In the cell below, print the length of the text.

<details><summary> Click to see final code</summary>

```
print(f'Length of text: {len(text)} characters')
```
</details>

In [ ]:
# Add code to print length of text
print(f'Length of text: {len(text)} characters')

Length of text: 2390691 characters


2. Add a print statement to the cell below to see how long the vocab is.

In [ ]:
# Sort the text to make a vocab
vocab = sorted(set(text))

# Add a print statement to print the length of the vocab
print(f'Length of vocab: {len(vocab)} vocabs')

Length of vocab: 86 vocabs


##Vectorizing

In order to put the text data into your network, you will need to convert letters into numbers. You already have a vocab created that contains all the characters in your training data. In order to understand how the tokenization process will work, follow the instructions below to tokenize some example text.

1. Add a print statement to print the chars.
2. Run the cell

In [ ]:
# Create some example text
example_texts = ['abcdefg', 'xyz']

# Split the text into characters
chars = tf.strings.unicode_split(example_texts, input_encoding='UTF-8')

# Add your print statement
print(chars)

<tf.RaggedTensor [[b'a', b'b', b'c', b'd', b'e', b'f', b'g'], [b'x', b'y', b'z']]>


3. Look at the code that creates a string lookup layer to convert characters into ids
4. Add code to print the ids and run the cell




In [ ]:
# Layer that converts chars to ids
ids_from_chars = tf.keras.layers.StringLookup(vocabulary=list(vocab), mask_token=None)

# Create ids from chars using your new layer
ids = ids_from_chars(chars)

# Print the ids
print(ids)

<tf.RaggedTensor [[60, 61, 62, 63, 64, 65, 66], [83, 84, 85]]>


5. Create another layer that converts the ids back into characters.

6. Add code to print the chars



In [ ]:
# Layer to convert the ids back into chars
chars_from_ids = tf.keras.layers.StringLookup(vocabulary=ids_from_chars.get_vocabulary(), invert=True, mask_token=None)
# Get characters from ids
chars = chars_from_ids(ids)

# Add your code to print chars
print(chars)

<tf.RaggedTensor [[b'a', b'b', b'c', b'd', b'e', b'f', b'g'], [b'x', b'y', b'z']]>


7. Create a function that gets text from the ids
8. Add code to print the text

In [ ]:
def text_from_ids(ids):
  return tf.strings.reduce_join(chars_from_ids(ids), axis=-1)

# Add code to print the text_from_ids(ids)
print(text_from_ids(ids))

tf.Tensor([b'abcdefg' b'xyz'], shape=(2,), dtype=string)


##Creating the Training Data

Now that you know how to vectorize text, it's time to create the samples for your network.

Your prediction task for this network is what should the next character be. You will be creating training examples so for each input sequence the goal output will be the input sequence moved over one character.

For example, let's say that your training data is a text file containing the text "Hello World! I am a potato."

![A representation of your data textfile.](https://i.imgur.com/3GZvFTB.png)

Your first task is to split the text into sequences. You will pick a sequence size and then break the data into small chunks. Because of how the input and labels will work for this network, you will want your chunks to be one character larger than the sequence size.

For example, using the input "Hello World! I am a potato." and the sequence size of 10, your data would be split into chunks of 11 characters. It will look like this.

![the text split into chunks](https://i.imgur.com/mBSnFXZ.png)

###Divide Text into Sequences

Look at the cell below to see how you can split the text into ids.

1. Add code to print all the ids.
2. Run the cell to see the results.

In [ ]:
all_ids = ids_from_chars(tf.strings.unicode_split(text, 'UTF-8'))
# Add code to print all ids
print(all_ids)

tf.Tensor([45 74 15 ... 16  2  2], shape=(2390691,), dtype=int64)



Now that you havve ids, you can create a tensorflow Dataset from them.

3. Run the cell to create a dataset

In [ ]:
ids_dataset = tf.data.Dataset.from_tensor_slices(all_ids)


4. Run the cell below to see the first 10 characters of your data set.

In [ ]:
for ids in ids_dataset.take(10):
    print(chars_from_ids(ids).numpy().decode('utf-8'))

N
o
-
B
a
k
e
 
N
u


5. Now that you have a dataset, set the `seq_length` variable to the sequence length. 100 will be a good size for this data, but you might want to experiment with this later.

In [ ]:
seq_length = 100

6. Next, run the cell to set how many examples you want to run per epoch. You will set this to the length of the text divided by the seq_length (rounded down).

In [ ]:
examples_per_epoch = len(text)//(seq_length+1)

7. Run the cell below to divide the data into sequences

In [ ]:
sequences = ids_dataset.batch(seq_length+1, drop_remainder=True)

8. Add a print statement inside the for loop to print the chars from the sequence.
9. Run the cell to look at the chars in the first sequence

In [ ]:
for seq in sequences.take(1):
    # Add a print statement here
    print(seq)

tf.Tensor(
[45 74 15 33 60 70 64  3 45 80 79  3 34 74 74 70 68 64 78  2 19  3 62 16
  3 65 68 77 72 71 84  3 75 60 62 70 64 63  3 61 77 74 82 73  3 78 80 66
 60 77  2  3 19 17 20  3 62 16  3 64 81 60 75 74 77 60 79 64 63  3 72 68
 71 70  2  3 19 17 20  3 79 78 75 16  3 81 60 73 68 71 71 60  2  3 19 17
 20  3 62 16  3], shape=(101,), dtype=int64)


10. Create a for loop with a print statement inside to print the text from five sequences.

In [ ]:
# Add your loop and print statement here
for seq in sequences.take(5):
  print(seq)

tf.Tensor(
[45 74 15 33 60 70 64  3 45 80 79  3 34 74 74 70 68 64 78  2 19  3 62 16
  3 65 68 77 72 71 84  3 75 60 62 70 64 63  3 61 77 74 82 73  3 78 80 66
 60 77  2  3 19 17 20  3 62 16  3 64 81 60 75 74 77 60 79 64 63  3 72 68
 71 70  2  3 19 17 20  3 79 78 75 16  3 81 60 73 68 71 71 60  2  3 19 17
 20  3 62 16  3], shape=(101,), dtype=int64)
tf.Tensor(
[61 77 74 70 64 73  3 73 80 79 78  3 10 75 64 62 60 73 78 11  2  3 20  3
 51 61 78 75 16  3 61 80 79 79 64 77  3 74 77  3 72 60 77 66 60 77 68 73
 64  2  3 21  3 19 17 20  3 62 16  3 61 68 79 64  3 78 68 85 64  3 78 67
 77 64 63 63 64 63  3 77 68 62 64  3 61 68 78 62 80 68 79 78 40 73  3 60
  3 67 64 60 81], shape=(101,), dtype=int64)
tf.Tensor(
[84  3 20 15 76 80 60 77 79  3 78 60 80 62 64 75 60 73 14  3 72 68 83  3
 61 77 74 82 73  3 78 80 66 60 77 14  3 73 80 79 78 14  3 64 81 60 75 74
 77 60 79 64 63  3 72 68 71 70  3 60 73 63  3 61 80 79 79 64 77  3 74 77
  3 72 60 77 66 60 77 68 73 64 16  2  3 50 79 68 77  3 74 81 64 77  3 72
 

###Create Input and Target Pairs


Just like with a classification problem, you need input and a label. For your MNIST data, your input was an image and the label was what number the image represented.

In this case, your input will be the first 100 characters of your sequence, and the label will be the last 100 characters of your sequence. Let's look at our Hello World example to see this in action.

If the text you are starting with is "Hello World! I am a potato.", and the sequence size is 10, then the first sequence will be "Hello World". When you split it into an input and a label, the first 10 characters would be the input ("Hello Worl") and the last 10 characters would be the label ("ello World").

The goal is to get the network to predict that the output is "ello World". Your target output/label, in this case, is "ello World"

![in input and target pair for the string "Hello World!"](https://i.imgur.com/9YGEaPv.png)

Follow the steps to divide your data into input and target sequences.

1. Run the cell below to create a function that will split a sequence into input text and target text.



In [ ]:
def split_input_target(sequence):
  input_text = sequence[:-1]
  target_text = sequence[1:]
  return input_text, target_text

2. Create some example text
3. Run the cell to print out the split input and target lists.

In [ ]:
example_text = 'Carol is studying at cornell'
print(split_input_target(list(example_text)))

(['C', 'a', 'r', 'o', 'l', ' ', 'i', 's', ' ', 's', 't', 'u', 'd', 'y', 'i', 'n', 'g', ' ', 'a', 't', ' ', 'c', 'o', 'r', 'n', 'e', 'l'], ['a', 'r', 'o', 'l', ' ', 'i', 's', ' ', 's', 't', 'u', 'd', 'y', 'i', 'n', 'g', ' ', 'a', 't', ' ', 'c', 'o', 'r', 'n', 'e', 'l', 'l'])


4. Run the cell to create a dataset using all the sequences you created before and applying your `split_input_target` function.

In [ ]:
dataset = sequences.map(split_input_target)

5. Run the cell to see the input and target for an example from the dataset.

In [ ]:
for input_example, target_example in dataset.take(1):
  print("Input :", text_from_ids(input_example).numpy())
  print("Target:", text_from_ids(target_example).numpy())

Input : b'No-Bake Nut Cookies\n1 c. firmly packed brown sugar\n 1/2 c. evaporated milk\n 1/2 tsp. vanilla\n 1/2 c.'
Target: b'o-Bake Nut Cookies\n1 c. firmly packed brown sugar\n 1/2 c. evaporated milk\n 1/2 tsp. vanilla\n 1/2 c. '


5. Set a batch size of 64
6. Set a buffer size of 10000
7. Run the cell to set up and print your final dataset.

In [ ]:
# Batch size
BATCH_SIZE = 64

# Buffer size
BUFFER_SIZE = 10000

# Set up a dataset
dataset = (dataset
           .shuffle(BUFFER_SIZE)
            .batch(BATCH_SIZE, drop_remainder=True)
             .prefetch(tf.data.experimental.AUTOTUNE))

print(dataset)

<_PrefetchDataset element_spec=(TensorSpec(shape=(64, 100), dtype=tf.int64, name=None), TensorSpec(shape=(64, 100), dtype=tf.int64, name=None))>


#Relational Neural Networks

Now that you have a dataset ready to be used to train a network, it's time to start building your network. In order to do text generation, you will be using a Relational Neural Network, or an RNN.

**RNNs** are a kind of network that are specially positioned to deal with sequences of data.

Your network will maintain an internal state that gets updated with each sequence that gets added to the network. This way, it will "remember" the input that came before it. Each time you run the network, you will be giving it input data, and the previous state of the network. It will use this information to update its state, and also predict what the next input should be.

These networks are very powerful, and you are about to see them in action!

##Building Your Network

Now it's time to start building your RNN. In the past, we have simply put together a sequential model by adding layers to it. However, since this network works a little differently, you will be setting up the structure of the network and also creating a way to maintain its internal state at the same time.

This means that you will be creating a class called **MyModel** that will both set up the structure of your RNN and create a **call** function that will help update the network with the addition of a new character.

To start, set a few key variables that will be used when you create your network.

1. Set `vocab_size` to the length of the vocab.
2. Create an `emmbedding_dim` variable to set how many different dimensions (256) the network should look at the data with.
3. Finally, set the units of the rnn to be 1024


<details><summary>Click for finished code</summary>

```
vocab_size = len(vocab)
embedding_dim = 256
rnn_units = 1024

```
</details>

In [ ]:
vocab_size = len(vocab)
embedding_dim = 256
rnn_units = 1024

4. Look inside the init function to learn about what each layer is for.

  **Layers breakdown**

  * The first layer is an embedding layer that helps read in the input data. This is similar to the input layers you have created for other networks.

  * The gru layer is the layer that will be doing a lot of the calculations for this network. You can see that it has a certain amount of units, and will return it's an internal state.

  * The last layer is the output layer and it is a Dense layer.

5. Look at the call method that will be used to update the internal state of the network.

6. In the last line, set the embedding_dim and rnn_units to the variables you created above.
7. Run the cell to create a model.

In [ ]:
class MyModel(tf.keras.Model):

  def __init__(self, vocab_size, embedding_dim, rnn_units):
    super().__init__(self)

    self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
    self.gru = tf.keras.layers.GRU(rnn_units, return_sequences=True, return_state=True)
    self.dense = tf.keras.layers.Dense(vocab_size)


  def call(self, inputs, states=None, return_state=False, training=False):
    x = inputs
    x = self.embedding(x, training=training)
    if states is None:
      states = self.gru.get_initial_state(x)
    x, states = self.gru(x, initial_state=states, training=training)
    x = self.dense(x, training=training)
    if return_state:
      return x, states
    else:
      return x


model = MyModel(vocab_size=len(ids_from_chars.get_vocabulary()),
                # Set the embedding dim
                embedding_dim= embedding_dim,
                # Set the rnn units
                rnn_units= rnn_units)


8. Run the cell below to create an example batch using a random input example batch and target example batch from the dataset.
9. Look at the output to see what the batch size, sequence length, and vocab size is.

In [ ]:
for input_example_batch, target_example_batch in dataset.take(1):
  example_batch_predictions = model(input_example_batch)
  print(example_batch_predictions.shape, "# (batch_size, sequence_length, vocab_size)")

(64, 100, 87) # (batch_size, sequence_length, vocab_size)


10. Add code to print a summary of the model.

<details><summary>Click to see code</summary>

```
model.summary()
```
</details>

In [ ]:
# Print a model summary
model.summary()

Model: "my_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       multiple                  22272     
                                                                 
 gru (GRU)                   multiple                  3938304   
                                                                 
 dense (Dense)               multiple                  89175     
                                                                 
Total params: 4,049,751
Trainable params: 4,049,751
Non-trainable params: 0
_________________________________________________________________


###Random Example


Just to see how the network will function, take a random sample of the data, and see how it could function.

1. In the cell below, add a print statement to print the `sampled_indices`

In [ ]:
sampled_indices = tf.random.categorical(example_batch_predictions[0], num_samples=1)
sampled_indices = tf.squeeze(sampled_indices, axis=-1).numpy()

# Add a print statement to print sampled_indices
print(sampled_indices)

[68 85 15 72  4 25 79 28 11 18 52 55 68 29 36 30 21  3 85 64 53 77 79  8
 46 63 30 75 66 56  7 55  2 12  1  5 70 31 20 53 79 23 28 27 73 49 40 76
  0 36 33 68 15 74 46 67 55 37 13 58 59 69 83 56 26  4 38  7 84 47 42 82
 32 64  8 46  0 86 83 25 58 40  8 38 20 44 45 81 12 73 18 12 13 80 60 43
 54 48 34  3]


2. Run the cell below to see what the input and next character predictions are for that sample.

In [ ]:
# View the input and predictions for the sample
print("Input:\n", text_from_ids(input_example_batch[0]).numpy())
print("Next Char Predictions:\n", text_from_ids(sampled_indices).numpy())

Input:
 b'eat until golden yellow.\n Beat together eggs, half and half, flour, salt, pepper and nutmeg, then ad'
Next Char Predictions:
 b'iz-m!7t:)0UXi;E=3 zeVrt&Od=pgY%X\n*\t"k?2Vt5:9nRIq[UNK]EBi-oOhXF+\\`jxY8!G%yPKwAe&O[UNK]~x7\\I&G2MNv*n0*+uaLWQC '


As you can see, the predictions are nonsense. However, the output is the right amount of text. This tells you that the structure of the network is working, but since it is not trained the output is nonsense. Now it's time to train the network.

#Training



Since you know the structure of your network works, it's time to train it. Now that you have input and target outputs, you are essentially just looking at a classification problem. You have input in the form of the internal state and character, and the task is to predict the next character.

Since this is a classification problem, a lot of the process will be familiar. The main difference will be the use of checkpoints. You will use checkpoints to save the model at different stages of the training. This way if the training fails, you will be able to restart it from a previous stage of training instead of having to start over.

1. Look at the line where the loss function gets set to Sparse Categorical Crossentropy
2. Add a print statement to print the shape of the `example_batch_predictions`
3. Add a print statement to print out the `example_batch_mean_loss`
4. Run the cell. If the exponential of the average loss is close to the vocab size, then you know that your model is set up correctly.

In [ ]:
# The line to set the loss function
loss = tf.losses.SparseCategoricalCrossentropy(from_logits=True)
# Get an example batch mean loss
example_batch_mean_loss = loss(target_example_batch, example_batch_predictions)

# Add a print statement to print the shape of example_batch_predictions.
print(example_batch_predictions.shape)
print("^ # (batch_size, sequence_length, vocab_size)")

# Add a print statement to print example_batch_mean_loss
print(example_batch_mean_loss)

# Print the exponential of the average loss
print("Exponential of average loss: ", tf.exp(example_batch_mean_loss).numpy())

(64, 100, 87)
^ # (batch_size, sequence_length, vocab_size)
tf.Tensor(4.4669476, shape=(), dtype=float32)
Exponential of average loss:  87.09048


5. Compile the model.

In [ ]:
model.compile(optimizer='adam', loss=loss)

Since this training will take some time, you are going to save checkpoints throughout the training process.

6. Run the cell below to get them set up.

In [ ]:
checkpoint_dir = 'training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}")
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint( filepath=checkpoint_prefix, save_weights_only=True)


7. Set the number of epochs to 20.

In [ ]:
EPOCHS = 20

8. Run the cell to start the training process.

In [ ]:
history = model.fit(dataset, epochs=EPOCHS, callbacks=[checkpoint_callback])

Epoch 1/20
369/369 [==============================] - 27s 58ms/step - loss: 2.0303
Epoch 2/20
369/369 [==============================] - 22s 55ms/step - loss: 1.0177
Epoch 3/20
369/369 [==============================] - 21s 52ms/step - loss: 0.8616
Epoch 4/20
369/369 [==============================] - 22s 54ms/step - loss: 0.7946
Epoch 5/20
369/369 [==============================] - 22s 54ms/step - loss: 0.7511
Epoch 6/20
369/369 [==============================] - 22s 54ms/step - loss: 0.7179
Epoch 7/20
369/369 [==============================] - 22s 56ms/step - loss: 0.6888
Epoch 8/20
369/369 [==============================] - 23s 57ms/step - loss: 0.6636
Epoch 9/20
369/369 [==============================] - 22s 55ms/step - loss: 0.6396
Epoch 10/20
369/369 [==============================] - 23s 56ms/step - loss: 0.6177
Epoch 11/20
369/369 [==============================] - 23s 57ms/step - loss: 0.5974
Epoch 12/20
369/369 [==============================] - 23s 57ms/step - loss: 0.5788
E

#### **Note**: The rest of this notebook will be your assignment for this unit. Complete the rest of the work, and then submit this notebook for a grade.

#Generate Text

Now that your model is trained, it's time to actually use it to generate text. Each time you call the model, you give it an "internal state" and some text. The model returns a prediction for the next character and an updated state that you can use to put back into the network to predict

##Define a One Step Model

This is what each "step" of the prediction looks like. In order to generate text, you will create a new class that uses the model to predict new text.

1. Read through the code and comments below to learn more about what makes up a one step model.

In [ ]:
class OneStep(tf.keras.Model):
  def __init__(self, model, chars_from_ids, ids_from_chars, temperature=1.0):
    super().__init__()
    # Set your arguments
    self.temperature = temperature
    self.model = model
    self.chars_from_ids = chars_from_ids
    self.ids_from_chars = ids_from_chars

    # Create a mask to prevent "UNK" from being generated
    skip_ids = self.ids_from_chars(['[UNK]'])[:, None]
    sparse_mask = tf.SparseTensor(
      #Put an -inf at each bad index
      values=[-float('inf')]*len(skip_ids),
      indices=skip_ids,
      # Match the shape to the vocabulary
      dense_shape=[len(ids_from_chars.get_vocabulary())])

    #Add the prediction mask to the model
    self.prediction_mask = tf.sparse.to_dense(sparse_mask)

  @tf.function
  def generate_one_step(self, inputs, states=None):

    # Convert strings into token ids
    input_chars = tf.strings.unicode_split(inputs, 'UTF-8')
    input_ids = self.ids_from_chars(input_chars).to_tensor()

    # Use the rnn model on the inputs
    predicted_logits, states = self.model(inputs=input_ids, states=states, return_state=True)

    # Only use the last prediction
    predicted_logits = predicted_logits[:, -1, :]

    # Adjust using temperature
    predicted_logits = predicted_logits/self.temperature

    # Apply your prediction mask
    predicted_logits = predicted_logits

    # Sample the output to generate ids
    predicted_ids = tf.random.categorical(predicted_logits, num_samples=1)
    predicted_ids = tf.squeeze(predicted_ids, axis=-1)

    # Convert from ids back to chars
    predicted_chars = self.chars_from_ids(predicted_ids)

    # Return the predicted chars and states
    return predicted_chars, states

# Generate a "one step" model using this class and your model from above
one_step_model = OneStep(model, chars_from_ids, ids_from_chars)

2. Initialize states to None
3. Change the string in the next_char variable to the text you want the network to start with.
4. Run the cell.

In [ ]:
# Initialize the states to None
states = None
# Add to or change the string in the square brackets
next_char = tf.constant(['start string'])
result = [next_char]

5. Create a for loop with the amount of characters that you want to generate.
6. Inside the loop, append next_char to the result.

<details><summary> Click to see finished code</summary>

```
for n in range(1000):
  next_char, states =  one_step_model.generate_one_step(next_char, states=states)
  result.append(next_char)

```
</details>

In [ ]:
# Add a for loop for the amount of characters (1000 is a good number to start with)
for n in range(1000):
  # Leave this line untouched
  next_char, states =  one_step_model.generate_one_step(next_char, states=states)
  # Append next_char to result
  result.append(next_char)

7. Run the cell to take the results from the for loop and join it together to create a string


In [ ]:
result = tf.strings.join(result)

8. Run the cell to see your results.

In [ ]:
print(result[0].numpy().decode('utf-8'), '\n\n' + '_'*80)

start string half of the tomato sauce and 2 1/2 cans of water and place in a medium skillet.
 Cook carrots, put sticky.

Dreenn'S Banana-Cherry Jello Salad
2 (3 oz.) pkg. raspberry jello
 1 tsp. vanilla
 2 eggs
 4 tsp. baking powder
 1/2 tsp. salt
 1 tsp. baking powder
 pinch of peppermint (aside)
 2 small cans chickenMix condensed milk, egg, milk and oil together until light and fluffy.
 Add butter and marshmallows.
 Boil quickly.
 Drain.
 Add real thickness, grated onion, garlic powder and garbanzo beans (break forountil firm.
 Roll in butter, on the pie filling, prayer to combine the tomatoes and liquid.
 Pour browned beans evenly and bake for about 1 to 1 1/2 hours or until brown.
 Turn out on clean to when boiled, add water.
 Chill; coat well to set. Da kack add slightly softened butter.
 Mix well and add sugar, stirring well.
 Add eggs; beat well.
 Stir in raisins.
 Beat until well blended. (1 1/2 c. milk with juice
 1/3 cup sugar
 3 eggs
 1/2 c. milk
 1/2 c. chopped chicken brot

#Improving the Model


Now that you can use your model to generate text, you know that it works! You might have noticed, that some of the words that the model is generating aren't real words, or that the grammar isn't quite what you wanted.

This is your chance to improve the model. Can you think of some ways that you could improve the model?

1. Experiment with your model and see if you can improve it.
2. Look at the cell below for some code to use to do this.

###Some ideas:



*   Epochs: Does training the model for more epochs improve the model.
*   Start String: Experiment with the string to start the network and see how that affects the output.
*   Add a Layer: Add a RNN layer to improve the models accuracy.
*   Adjust Temperature: Adjust the temperature to reduce random predictions.


In [ ]:
# This code connects your Drive account with your notebook
from google.colab import drive
drive.mount('/content/drive')

# This command will copy the training_checkpoints folder from your notebook to your drive
!cp -r "/content/training_checkpoints" "/content/drive/MyDrive"

# This command copies the training_checkpoints folder back into your notebook from your drive
!cp -r "/content/drive/MyDrive/training_checkpoints" "/content/training_checkpoints"

# This code will run the training process again, using the training_checkpoints from before
# This way you can pick up training where you left off - so if you do 10 more epochs then
# your network will have trained for 30 total epochs.
history = model.fit(dataset, epochs=EPOCHS, callbacks=[checkpoint_callback])

Mounted at /content/drive
Epoch 1/20
369/369 [==============================] - 22s 54ms/step - loss: 0.5027
Epoch 2/20
369/369 [==============================] - 22s 54ms/step - loss: 0.5024
Epoch 3/20
369/369 [==============================] - 23s 54ms/step - loss: 0.5026
Epoch 4/20
369/369 [==============================] - 22s 56ms/step - loss: 0.5051
Epoch 5/20
369/369 [==============================] - 22s 56ms/step - loss: 0.5084
Epoch 6/20
369/369 [==============================] - 23s 57ms/step - loss: 0.5113
Epoch 7/20
369/369 [==============================] - 24s 57ms/step - loss: 0.5163
Epoch 8/20
369/369 [==============================] - 23s 57ms/step - loss: 0.5239
Epoch 9/20
369/369 [==============================] - 23s 57ms/step - loss: 0.5287
Epoch 10/20
369/369 [==============================] - 23s 57ms/step - loss: 0.5361
Epoch 11/20
369/369 [==============================] - 23s 57ms/step - loss: 0.5436
Epoch 12/20
369/369 [==============================] - 23s 

In [ ]:
# Add code to re-train your network
history = model.fit(dataset, epochs=EPOCHS, callbacks=[checkpoint_callback])


Epoch 1/20
369/369 [==============================] - 23s 55ms/step - loss: 0.6663
Epoch 2/20
369/369 [==============================] - 22s 54ms/step - loss: 0.6688
Epoch 3/20
369/369 [==============================] - 23s 54ms/step - loss: 0.6968
Epoch 4/20
369/369 [==============================] - 23s 55ms/step - loss: 0.7074
Epoch 5/20
369/369 [==============================] - 23s 57ms/step - loss: 0.7186
Epoch 6/20
369/369 [==============================] - 24s 57ms/step - loss: 0.7229
Epoch 7/20
369/369 [==============================] - 22s 56ms/step - loss: 0.7528
Epoch 8/20
369/369 [==============================] - 24s 56ms/step - loss: 0.8079
Epoch 9/20
369/369 [==============================] - 23s 57ms/step - loss: 0.8622
Epoch 10/20
369/369 [==============================] - 23s 56ms/step - loss: 0.9022
Epoch 11/20
369/369 [==============================] - 23s 56ms/step - loss: 1.0287
Epoch 12/20
369/369 [==============================] - 23s 56ms/step - loss: 1.6129
E

##Save and Export


Once you are happy with your model, you should export it so that you can reload it and use it.

1. Run the cell to save your one_step_model


In [ ]:
tf.saved_model.save(one_step_model, 'one_step')

2. Add code to copy the onestep model to your drive

<details><summary>Click to see finished code</summary>

```
!cp -r "/content/one_step" "/content/drive/MyDrive"
```
</details>

In [ ]:
# Add code to copy the one_step folder to MyDrive
!cp -r "/content/one_step" "/content/drive/MyDrive"

3. Run the cell to reload the model from the one_step folder

In [ ]:
one_step_reloaded = tf.saved_model.load('one_step')

4. Now add code to use the reloaded model to make predictions.

<details><summary>Click here to see finished code</summary>

```
states = None
next_char = tf.constant(['\n'])
result = [next_char]

for n in range(200):
  next_char, states = one_step_reloaded.generate_one_step(next_char, states=states)
  result.append(next_char)

print(tf.strings.join(result)[0].numpy().decode("utf-8"))
```
</details>

In [ ]:
# This code will be very similar to the code used for the text generation originally
states = None
next_char = tf.constant(['\n'])
result = [next_char]

for n in range(200):
  next_char, states = one_step_reloaded.generate_one_step(next_char, states=states)
  result.append(next_char)

print(tf.strings.join(result)[0].numpy().decode("utf-8"))



Cook OatsMix celored
 1 large par tos together until melt doole
2 to 2 cans or until gones and kitlly thours af hellow 1 cup water
 1 c. vinegar
 1/2 c. cooked macaroni
 1 c. light cream oven whipies


##Recap
Now that you can create a RNN to generate text, you can see the fruits of your labor. Creating a one step class allows you to do just one step of the network. You can put the one step in a loop and use that to generate text.


* Collect your own text data and use this structure to generate even more text
* Continue to work on your model and tweak the settings until you are satisfied with the output.